# ATLAS baseline fine-tuning (Colab)

This notebook runs a small LoRA supervised fine-tuning baseline and records resource measurements. It is not the ATLAS optimization algorithm and does not claim efficiency gains. Use only models and datasets you are permitted to access.

Validation loss is diagnostic, not a universal capability score. Compare matched runs with a task-specific held-out evaluator.

In [ ]:
%pip -q install 'torch>=2.4' 'transformers>=4.46,<5' 'datasets>=3,<5' 'peft>=0.13,<1' 'accelerate>=1,<2' 'safetensors>=0.4' 'psutil>=6,<8'


In [ ]:
import json, platform, time
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## Configure the run

Choose a small, permitted model and dataset. The dataset must contain the text field specified below. Do not put access tokens directly in the notebook.

In [ ]:
MODEL_ID = 'YOUR_OPEN_MODEL_ID'
DATASET_ID = 'YOUR_DATASET_ID'
DATASET_CONFIG = None
TEXT_FIELD = 'text'
MAX_LENGTH = 512
MAX_STEPS = 100
EVAL_STEPS = 25
SEED = 42
OUTPUT_DIR = '/content/atlas-output'
RUN_NAME = 'lora-baseline'

assert not MODEL_ID.startswith('YOUR_'), 'Set MODEL_ID first.'
assert not DATASET_ID.startswith('YOUR_'), 'Set DATASET_ID first.'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
dataset_args = {'path': DATASET_ID}
if DATASET_CONFIG:
    dataset_args['name'] = DATASET_CONFIG
raw = load_dataset(**dataset_args)
if 'train' in raw:
    train_raw = raw['train']
    if 'validation' in raw:
        eval_raw = raw['validation']
    elif 'test' in raw:
        eval_raw = raw['test']
    else:
        split = train_raw.train_test_split(test_size=0.05, seed=SEED)
        train_raw, eval_raw = split['train'], split['test']
else:
    split = raw.train_test_split(test_size=0.05, seed=SEED)
    train_raw, eval_raw = split['train'], split['test']
if TEXT_FIELD not in train_raw.column_names:
    raise ValueError(f'TEXT_FIELD={TEXT_FIELD!r} missing. Columns: {train_raw.column_names}')
print('Train examples:', len(train_raw), 'Eval examples:', len(eval_raw))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
def tokenize_batch(batch):
    return tokenizer([str(v) for v in batch[TEXT_FIELD]], truncation=True, max_length=MAX_LENGTH)
train_ds = train_raw.map(tokenize_batch, batched=True, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(tokenize_batch, batched=True, remove_columns=eval_raw.column_names)
model_kwargs = {'low_cpu_mem_usage': True}
if torch.cuda.is_available():
    model_kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.config.use_cache = False
lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05, target_modules='all-linear')
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
args = TrainingArguments(
    output_dir=OUTPUT_DIR, run_name=RUN_NAME, max_steps=MAX_STEPS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03,
    logging_steps=10, eval_strategy='steps', eval_steps=EVAL_STEPS,
    save_strategy='steps', save_steps=EVAL_STEPS, save_total_limit=2,
    gradient_checkpointing=True,
    bf16=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    fp16=bool(torch.cuda.is_available() and not torch.cuda.is_bf16_supported()),
    report_to='none', seed=SEED, data_seed=SEED, remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
train_output = trainer.train()
eval_metrics = trainer.evaluate()
elapsed_seconds = time.perf_counter() - start
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Elapsed seconds:', round(elapsed_seconds, 2))
print('Evaluation metrics:', eval_metrics)


In [ ]:
peak_memory_gb = None
gpu_name = None
gpu_count = 0
if torch.cuda.is_available():
    peak_memory_gb = round(torch.cuda.max_memory_allocated() / 1e9, 4)
    gpu_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
manifest = {
    'schema_version': 1, 'run_name': RUN_NAME, 'model_id': MODEL_ID,
    'dataset_id': DATASET_ID, 'seed': SEED, 'max_steps': MAX_STEPS,
    'max_length': MAX_LENGTH, 'lora_rank': 8,
    'elapsed_seconds': round(elapsed_seconds, 4), 'gpu_name': gpu_name,
    'gpu_count': gpu_count, 'peak_memory_gb': peak_memory_gb,
    'train_metrics': {k: float(v) for k, v in train_output.metrics.items() if isinstance(v, (int, float))},
    'evaluation_metrics': {k: float(v) for k, v in eval_metrics.items() if isinstance(v, (int, float))},
    'note': 'Record exact revisions, dataset fingerprint, code commit and task-specific held-out score before making comparative claims.'
}
manifest_path = Path(OUTPUT_DIR) / 'atlas-run-manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('Manifest saved to:', manifest_path)


## Before claiming efficiency

Run a baseline and candidate with matched data, training-token budget, hardware and evaluator. Include hyperparameter-search compute. Repeat final runs where feasible. Do not calculate a savings percentage unless measurements are comparable and verified.